### This draft version skips the feature scaling, and significative PCA components reduce up to only 5
- Applying scaling, the number of components to cover 95% of the data variance is 11
- If MinMaxScaler is not apply (raw data), the number of components is only 2

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib import gridspec

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/nasa-cmaps/CMaps/RUL_FD002.txt
/kaggle/input/nasa-cmaps/CMaps/test_FD003.txt
/kaggle/input/nasa-cmaps/CMaps/Damage Propagation Modeling.pdf
/kaggle/input/nasa-cmaps/CMaps/readme.txt
/kaggle/input/nasa-cmaps/CMaps/train_FD003.txt
/kaggle/input/nasa-cmaps/CMaps/test_FD004.txt
/kaggle/input/nasa-cmaps/CMaps/train_FD004.txt
/kaggle/input/nasa-cmaps/CMaps/x.txt
/kaggle/input/nasa-cmaps/CMaps/test_FD002.txt
/kaggle/input/nasa-cmaps/CMaps/train_FD001.txt
/kaggle/input/nasa-cmaps/CMaps/train_FD002.txt
/kaggle/input/nasa-cmaps/CMaps/RUL_FD001.txt
/kaggle/input/nasa-cmaps/CMaps/RUL_FD004.txt
/kaggle/input/nasa-cmaps/CMaps/RUL_FD003.txt
/kaggle/input/nasa-cmaps/CMaps/test_FD001.txt
/kaggle/input/nasa-cmaps/cmaps/CMaps/RUL_FD002.txt
/kaggle/input/nasa-cmaps/cmaps/CMaps/test_FD003.txt
/kaggle/input/nasa-cmaps/cmaps/CMaps/Damage Propagation Modeling.pdf
/kaggle/input/nasa-cmaps/cmaps/CMaps/readme.txt
/kaggle/input/nasa-cmaps/cmaps/CMaps/train_FD003.txt
/kaggle/input/nasa-cmaps/cmaps/CM

# Data description

In [2]:
train_FD001_no_name = pd.read_csv("../input/nasa-cmaps/CMaps/train_FD001.txt", sep = "\s+", header = None)
test_FD001_no_name = pd.read_csv("../input/nasa-cmaps/CMaps/test_FD001.txt", sep = "\s+", header = None)

This dataset has 26 columns (as well as the datasets for the other 3 scenarios). Because of Python's numbering convention, the columns are numbered from 0 to 25. Description of each column is as follows:

* `Column 1`: Corresponds to engine number (This column is indexed 0 because of Python's numbering convention)
* `Column 2`: Corresponds to cycle number. If engine 1 fails after 192 cycles, the entries of second column for engine 1 will go from 1 to 192. Similarly for other engines. 
* `Columns 3,4,5`: 3 operational settings
* `Columns 6-26`: 21 sensor measurements

**Note**: Hence, we will always refer to the first column as column 1 even though it is indexed as 0 in Python. Similarly for other columns.

In [3]:
# Let's add columns' names for better identification
columns = {0:'engineNumber',1:'cycleNumber',2:'opSetting1',3:'opSetting2',4:'opSetting3',5:'sensor1',6:'sensor2',
           7:'sensor3',8:'sensor4',9:'sensor5',10:'sensor6',11:'sensor7',12:'sensor8',13:'sensor9',14:'sensor10',
           15:'sensor11',16:'sensor12',17:'sensor13',18:'sensor14',19:'sensor15',20:'sensor16',
           21:'sensor17',22:'sensor18',23:'sensor19',24:'sensor20',25:'sensor21'}

columns_to_drop=['engineNumber','cycleNumber']

In [4]:
train_FD001 = train_FD001_no_name.rename(columns=columns)
test_FD001  = test_FD001_no_name.rename(columns=columns)

train_FD001.describe()
#test_FD001.describe()

,engineNumber,cycleNumber,opSetting1,opSetting2,opSetting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.0,20631.00,20631.000000,20631.000000,20631.000000,2.063100e+04,...,20631.000000,20631.000000,20631.000000,20631.000000,2.063100e+04,20631.000000,20631.0,20631.0,20631.000000,20631.000000
mean,51.506568,108.807862,-0.000009,0.000002,100.0,518.67,642.680934,1590.523119,1408.933782,1.462000e+01,...,521.413470,2388.096152,8143.752722,8.442146,3.000000e-02,393.210654,2388.0,100.0,38.816271,23.289705
std,29.227633,68.880990,0.002187,0.000293,0.0,0.00,0.500053,6.131150,9.000605,1.776400e-15,...,0.737553,0.071919,19.076176,0.037505,1.387812e-17,1.548763,0.0,0.0,0.180746,0.108251
min,1.000000,1.000000,-0.008700,-0.000600,100.0,518.67,641.210000,1571.040000,1382.250000,1.462000e+01,...,518.690000,2387.880000,8099.940000,8.324900,3.000000e-02,388.000000,2388.0,100.0,38.140000,22.894200
25%,26.000000,52.000000,-0.001500,-0.000200,100.0,518.67,642.325000,1586.260000,1402.360000,1.462000e+01,...,520.960000,2388.040000,8133.245000,8.414900,3.000000e-02,392.000000,2388.0,100.0,38.700000,23.221800
50%,52.000000,104.000000,0.000000,0.000000,100.0,518.67,642.640000,1590.100000,1408.040000,1.462000e+01,...,521.480000,2388.090000,8140.540000,8.438900,3.000000e-02,393.000000,2388.0,100.0,38.830000,23.297900
75%,77.000000,156.000000,0.001500,0.000300,100.0,518.67,643.000000,1594.380000,1414.555000,1.462000e+01,...,521.950000,2388.140000,8148.310000,8.465600,3.000000e-02,394.000000,2388.0,100.0,38.950000,23.366800
max,100.000000,362.000000,0.008700,0.000600,100.0,518.67,644.530000,1616.910000,1441.490000,1.462000e+01,...,523.380000,2388.560000,8293.720000,8.584800,3.000000e-02,400.000000,2388.0,100.0,39.430000,23.618400


# Dataset simplification

In [5]:
train_FD001 = train_FD001.drop(columns=columns_to_drop) #.values
test_FD001  = test_FD001.drop(columns=columns_to_drop)

# Data normalization

In [6]:
from sklearn import preprocessing

# Train dataset is scaled so that values are in the range 0 to 1
scaler = preprocessing.MinMaxScaler()
train_FD001_scaled = pd.DataFrame(scaler.fit_transform(train_FD001), 
                              columns=train_FD001.columns, 
                              index=train_FD001.index)
train_FD001_scaled.describe()

,opSetting1,opSetting2,opSetting3,sensor1,sensor2,sensor3,sensor4,sensor5,sensor6,sensor7,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
count,20631.000000,20631.000000,20631.0,20631.0,20631.000000,20631.000000,20631.000000,20631.0,20631.000000,20631.000000,...,20631.000000,20631.000000,20631.000000,20631.000000,20631.0,20631.000000,20631.0,20631.0,20631.000000,20631.000000
mean,0.499490,0.501959,0.0,0.0,0.443052,0.424746,0.450435,0.0,0.980321,0.566459,...,0.580697,0.317871,0.226095,0.451118,0.0,0.434221,0.0,0.0,0.524241,0.546127
std,0.125708,0.244218,0.0,0.0,0.150618,0.133664,0.151935,0.0,0.138898,0.142527,...,0.157261,0.105763,0.098442,0.144306,0.0,0.129064,0.0,0.0,0.140114,0.149476
min,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000
25%,0.413793,0.333333,0.0,0.0,0.335843,0.331807,0.339467,0.0,1.000000,0.476651,...,0.484009,0.235294,0.171870,0.346287,0.0,0.333333,0.0,0.0,0.434109,0.452361
50%,0.500000,0.500000,0.0,0.0,0.430723,0.415522,0.435348,0.0,1.000000,0.578100,...,0.594883,0.308824,0.209516,0.438630,0.0,0.416667,0.0,0.0,0.534884,0.557443
75%,0.586207,0.750000,0.0,0.0,0.539157,0.508829,0.545324,0.0,1.000000,0.669887,...,0.695096,0.382353,0.249613,0.541362,0.0,0.500000,0.0,0.0,0.627907,0.652582
max,1.000000,1.000000,0.0,0.0,1.000000,1.000000,1.000000,0.0,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,0.0,0.0,1.000000,1.000000


In [7]:
# Test dataset is scaled according to train dataset
test_FD001_scaled = pd.DataFrame(scaler.transform(test_FD001), 
                              columns=test_FD001.columns, 
                              index=test_FD001.index)
test_FD001_scaled.describe()

,opSetting1,opSetting2,opSetting3,sensor1,sensor2,sensor3,sensor4,sensor5,sensor6,sensor7,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
count,13096.000000,13096.000000,13096.0,13096.0,13096.000000,13096.000000,13096.000000,13096.0,13096.000000,13096.000000,...,13096.000000,13096.000000,13096.000000,13096.000000,13096.0,13096.000000,13096.0,13096.0,13096.000000,13096.000000
mean,0.499358,0.503532,0.0,0.0,0.381051,0.371903,0.379564,0.0,0.970067,0.629231,...,0.651967,0.280919,0.201299,0.388395,0.0,0.380969,0.0,0.0,0.583335,0.609697
std,0.126591,0.245025,0.0,0.0,0.120753,0.109075,0.112902,0.0,0.170408,0.109708,...,0.119323,0.083727,0.052578,0.111617,0.0,0.102798,0.0,0.0,0.109830,0.116156
min,0.028736,0.000000,0.0,0.0,-0.024096,-0.043601,0.036124,0.0,0.000000,0.165862,...,0.147122,0.014706,0.044174,0.030396,0.0,0.083333,0.0,0.0,0.131783,0.056890
25%,0.413793,0.333333,0.0,0.0,0.297440,0.295618,0.298785,0.0,1.000000,0.557166,...,0.573561,0.220588,0.167045,0.310504,0.0,0.333333,0.0,0.0,0.511628,0.534935
50%,0.500000,0.500000,0.0,0.0,0.376506,0.369523,0.374578,0.0,1.000000,0.636071,...,0.658849,0.279412,0.198421,0.384763,0.0,0.416667,0.0,0.0,0.589147,0.614471
75%,0.586207,0.750000,0.0,0.0,0.460843,0.443046,0.452397,0.0,1.000000,0.706924,...,0.737740,0.338235,0.229229,0.459407,0.0,0.416667,0.0,0.0,0.658915,0.689589
max,0.948276,1.083333,0.0,0.0,0.930723,0.795945,0.862762,0.0,1.000000,0.964573,...,1.081023,0.647059,0.622046,0.833013,0.0,0.750000,0.0,0.0,0.984496,1.032450


# PCA model: Principal Components analysis
## Applying data scaling

In [8]:
from sklearn.decomposition import PCA

n_components = 24 # How many dimensions you want to reduce to
pca = PCA(n_components=n_components, svd_solver= 'full')

In [9]:
# Compute all PCA components FOR THE TRAINING SET
train_FD001_PCA = pca.fit_transform(train_FD001_scaled)
train_FD001_PCA = pd.DataFrame(train_FD001_PCA)
train_FD001_PCA.index = train_FD001_scaled.index

# Project the TEST SET onto the PCA space
test_FD001_PCA = pca.transform(test_FD001_scaled)
test_FD001_PCA = pd.DataFrame(test_FD001_PCA)
test_FD001_PCA.index = test_FD001_scaled.index

### Explained variance by PCA components
How many components do you need to explain >95% of the variance?

In [10]:
np.set_printoptions(precision=3, suppress=True) # 3 decimal places and don't use scientific notation

print(pca.explained_variance_ratio_)

[0.51  0.17  0.061 0.053 0.045 0.023 0.021 0.019 0.018 0.017 0.015 0.012
 0.012 0.011 0.007 0.006 0.001 0.    0.    0.    0.    0.    0.    0.   ]


**SOLUTION:**
 - **15 components are needed to explain 99%** of the total variance:
`1 - 0.001 - 0.006`  => 99.3%

- **11 components are needed to explain 95%** of the total variance:
`1 - 0.001 - 0.006 - 0.007 - 0.011 - 0.012 - 0.012`  => 95.1%

In [11]:
# In terms of absolute variance
print(pca.explained_variance_)

[0.179 0.06  0.022 0.018 0.016 0.008 0.007 0.007 0.006 0.006 0.005 0.004
 0.004 0.004 0.002 0.002 0.    0.    0.    0.    0.    0.    0.    0.   ]


## Applying data scaling

In [12]:
# Compute all PCA components FOR THE scaled TRAINING SET
train_FD001_PCA = pca.fit_transform(train_FD001)
train_FD001_PCA = pd.DataFrame(train_FD001_PCA)
train_FD001_PCA.index = train_FD001_scaled.index

# Project the scaled TEST SET onto the PCA space
test_FD001_PCA = pca.transform(test_FD001)
test_FD001_PCA = pd.DataFrame(test_FD001_PCA)
test_FD001_PCA.index = test_FD001_scaled.index

In [13]:
print(pca.explained_variance_ratio_)

[0.868 0.101 0.016 0.013 0.001 0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]


**SOLUTION:**
 - **4 components are needed to explain 99%** of the total variance:
`0.868 + 10.1 + 0.016 + 0.013`  => 99.8%

- **2 components are needed to explain 95%** of the total variance:
`0.868 + 10.1`  => 96.9%